In [2]:
import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

from data.contracts import active_contracts
from data.processing.usdinr.constants import CHALLENGER50, BASE46
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.labels import add_triple_barrier_labels

print(active_contracts())

['26OCTFUT', '26NOVFUT']


In [3]:
from data.processing.usdinr.constants import CORE31, ADDED6, CONTROLS9, CHALLENGERS4, BASE46, CHALLENGER50

print(f"CORE31:       {len(CORE31)}")
print(f"ADDED6:       {len(ADDED6)}")
print(f"CONTROLS9:    {len(CONTROLS9)}")
print(f"CHALLENGERS4: {len(CHALLENGERS4)}")
print(f"BASE46:       {len(BASE46)}")
print(f"CHALLENGER50: {len(CHALLENGER50)}")

CORE31:       31
ADDED6:       6
CONTROLS9:    9
CHALLENGERS4: 4
BASE46:       46
CHALLENGER50: 50


In [5]:
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.labels import add_triple_barrier_labels, label_summary

db = DuckDBClient()
s  = S3Client()

builder = USDINRFeatureBuilder(
    storage   = s,
    reader    = db,
    contracts = ["26OCTFUT", "26NOVFUT"],
    auto_build = False,         # don't build yet
)

# Load raw data
df_raw = builder.load_data()
print(f"Raw: {len(df_raw):,} rows, {df_raw['symbol'].unique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw: 317,121 rows, ['USDINR26OCTFUT']


In [6]:
df_reg = builder.preprocess()
print(f"Regularized: {len(df_reg):,} rows")
print(f"Columns: {list(df_reg.columns)}")
print(df_reg.head(3))

Regularized: 530,099 rows
Columns: ['ts_sec', 'symbol', 'open', 'high', 'low', 'close', 'volume_delta', 'tick_count', 'vwap', 'oi', 'spread_mean', 'spread_max', 'spread_bps', 'imbalance_mean', 'imbalance_std', 'imbalance_last', 'total_bid_qty', 'total_ask_qty', 'weighted_mid', 'price_impact', 'bid_p1', 'bid_q1', 'bid_p2', 'bid_q2', 'bid_p3', 'bid_q3', 'bid_p4', 'bid_q4', 'bid_p5', 'bid_q5', 'ask_p1', 'ask_q1', 'ask_p2', 'ask_q2', 'ask_p3', 'ask_q3', 'ask_p4', 'ask_q4', 'ask_p5', 'ask_q5', 'realized_vol_10s', 'realized_vol_30s', 'realized_vol_60s', 'realized_vol_300s', 'imbalance_ma_10s', 'imbalance_ma_30s', 'imbalance_ma_60s', 'spread_zscore', 'volume_ratio', 'price_mom_10s', 'price_mom_30s', 'price_mom_60s', 'ts_ist', 'ist_sec', 'date', 'is_observed', 'had_update']
       ts_sec          symbol     open  high      low  close  volume_delta  \
0  1790739000  USDINR26OCTFUT  96.2725  96.3  96.2725   96.3           1.0   
1  1790739001  USDINR26OCTFUT  96.3000  96.3  96.3000   96.3       

In [7]:
df_features = builder.calculate_features()
print(f"Features: {len(df_features):,} rows, {len(df_features.columns)} columns")

# Check all 50 features are present and see null rates
from data.processing.usdinr.constants import CHALLENGER50
null_rates = df_features[CHALLENGER50].isnull().mean().sort_values(ascending=False)
print(null_rates.head(20))

C:\projects\hedge_fund\data\processing\usdinr\features.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda day: day["ret_1s"].rolling(300, min_periods=120).corr(day["_ret_lag1"]))


Features: 530,099 rows, 107 columns
ask_shock_age              0.311314
sign_persistence_900s      0.071072
ofi_response_gap_z         0.044086
spread_depth_stress_900    0.028996
impact_r2_1800s            0.022581
impact_lambda_1800s        0.022581
semivar_imbalance_900s     0.010717
spread_rank_1800s          0.010717
ret_acf_lag1_300s          0.009730
log_vr_30_300s             0.005341
log_vr_5_300s              0.004444
jump_ratio_300s            0.004337
semivar_imbalance_300s     0.004265
range_position_300s        0.004265
log_rv_300s_new            0.003584
rv_ratio_60_300_new        0.003584
ofi_rate_diff_30_300_sl    0.003584
update_rate_300s           0.003548
ofi_activity_300s          0.003548
relative_migration_60      0.002151
dtype: float64


In [8]:
from data.processing.usdinr.labels import add_triple_barrier_labels, label_summary

df_labeled = add_triple_barrier_labels(
    df_features,
    horizon   = 1800,   # 30 min
    threshold = 10.0,   # ±10 bps
)

print(label_summary(df_labeled))
print(f"\nLabel distribution:")
print(df_labeled["label"].value_counts(normalize=True).sort_index())

           symbol  n_total  n_labeled   frac_up  frac_neutral  frac_down  \
0  USDINR26OCTFUT   530099     495899  0.067512      0.875678    0.05681   

   frac_unlabeled  
0        0.064516  

Label distribution:
label
-1.0    0.056810
 0.0    0.875678
 1.0    0.067512
Name: proportion, dtype: float64


In [10]:
# from data.processing.usdinr.labels import add_triple_barrier_labels

# horizons   = [300, 600, 900, 1800, 3600]
# thresholds = [5.0, 7.5, 10.0, 15.0, 20.0]

# rows = []
# for h in horizons:
#     for t in thresholds:
#         tmp = add_triple_barrier_labels(df_features, horizon=h, threshold=t)
#         lbl = tmp["label"].dropna()
#         n   = len(lbl)
#         rows.append({
#             "horizon_min":   h // 60,
#             "threshold_bps": t,
#             "pct_up":        round((lbl == 1).sum() / n * 100, 1),
#             "pct_neutral":   round((lbl == 0).sum() / n * 100, 1),
#             "pct_down":      round((lbl == -1).sum() / n * 100, 1),
#         })

# results = pd.DataFrame(rows)
# print(results.to_string(index=False))

In [11]:
"""
USDINR LightGBM Baseline — Return Forecasting
==============================================
Targets  : ret_60s, ret_300s (log-return in bps)
Features : BASE46  (CORE31 + ADDED6 + CONTROLS9)
Split    : Days 1-15 train | Days 16-20 val | Days 21-25 test
Outputs  : results/lgbm_baseline/metrics.json
           results/lgbm_baseline/feature_importance.csv
           results/lgbm_baseline/oof_predictions.csv
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.contracts import active_contracts
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import BASE46, CHALLENGER50

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR = pathlib.Path(r"C:\projects\hedge_fund\results\lgbm_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES  = BASE46          # 46 features
HORIZONS  = [60, 300]       # seconds
TRAIN_DAYS = 15
VAL_DAYS   = 5
# remaining days → test

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "verbosity":        -1,
    "boosting_type":    "gbdt",
    "num_leaves":       31,
    "learning_rate":    0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "min_child_samples": 100,
    "n_estimators":     1000,
    "early_stopping_rounds": 50,
}

# ---------------------------------------------------------------------------
# Step 1 — Load and build features
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading data and building features ===")

db = DuckDBClient()
s  = S3Client()

contracts = active_contracts()
log.info("Active contracts: %s", contracts)

# Also load prior month if available (for more training data)
all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]

all_frames = []
for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df["symbol"] = sym
            all_frames.append(df)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded — check S3 paths and contracts")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw data: %d rows across symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df = builder.dataset.copy()

log.info("Feature dataset: %d rows, %d columns", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Build forward return targets
# ---------------------------------------------------------------------------

log.info("=== Step 3: Building forward return targets ===")

df = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)

# Mid price
if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0

df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)

for h in HORIZONS:
    col = f"ret_{h}s"
    df[col] = (
        g["log_mid"].shift(-h) - df["log_mid"]
    ) * 1e4   # in basis points
    log.info("Target %s: %.2f%% non-null", col,
             df[col].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 4 — Train / val / test split by calendar day
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting by calendar day ===")

all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total trading days: %d", n_days)
log.info("Dates: %s ... %s", all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(
        f"Only {n_days} days available — need at least {TRAIN_DAYS + VAL_DAYS + 1}"
    )

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]

log.info("Train: %d days (%s → %s)", len(train_dates), train_dates[0],  train_dates[-1])
log.info("Val:   %d days (%s → %s)", len(val_dates),   val_dates[0],    val_dates[-1])
log.info("Test:  %d days (%s → %s)", len(test_dates),  test_dates[0],   test_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 5 — Train one model per horizon
# ---------------------------------------------------------------------------

log.info("=== Step 5: Training LightGBM models ===")

all_metrics      = {}
all_importance   = {}
oof_predictions  = df[["symbol", "ts_sec", "date"]].copy()

for horizon in HORIZONS:
    target = f"ret_{horizon}s"
    log.info("--- Horizon: %ds  Target: %s ---", horizon, target)

    # Drop rows where target or any feature is null
    valid = df[FEATURES + [target]].notna().all(axis=1)
    log.info("Valid rows: %d / %d (%.1f%%)",
             valid.sum(), len(df), valid.mean() * 100)

    tr = df[mask_train & valid]
    va = df[mask_val   & valid]
    te = df[mask_test  & valid]

    log.info("Train=%d  Val=%d  Test=%d", len(tr), len(va), len(te))

    X_tr, y_tr = tr[FEATURES].values, tr[target].values
    X_va, y_va = va[FEATURES].values, va[target].values
    X_te, y_te = te[FEATURES].values, te[target].values

    ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURES)
    ds_va = lgb.Dataset(X_va, label=y_va, feature_name=FEATURES, reference=ds_tr)

    params = {k: v for k, v in LGB_PARAMS.items()
              if k not in ("n_estimators", "early_stopping_rounds")}

    callbacks = [
        lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
        lgb.log_evaluation(100),
    ]

    model = lgb.train(
        params,
        ds_tr,
        num_boost_round      = LGB_PARAMS["n_estimators"],
        valid_sets           = [ds_va],
        callbacks            = callbacks,
    )

    log.info("Best iteration: %d", model.best_iteration)

    # Predictions
    pred_tr = model.predict(X_tr)
    pred_va = model.predict(X_va)
    pred_te = model.predict(X_te)

    def compute_metrics(y_true, y_pred, split_name):
        rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        ic, _ = spearmanr(y_true, y_pred)
        hit   = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
        log.info("%s | RMSE=%.4f  IC=%.4f  HitRate=%.4f", split_name, rmse, ic, hit)
        return {"rmse": rmse, "ic": ic, "hit_rate": hit, "n": len(y_true)}

    metrics = {
        "train": compute_metrics(y_tr, pred_tr, f"{target}/train"),
        "val":   compute_metrics(y_va, pred_va, f"{target}/val"),
        "test":  compute_metrics(y_te, pred_te, f"{target}/test"),
        "best_iteration": model.best_iteration,
        "n_features": len(FEATURES),
    }
    all_metrics[target] = metrics

    # Feature importance
    imp = pd.DataFrame({
        "feature":    FEATURES,
        "gain":       model.feature_importance("gain"),
        "split":      model.feature_importance("split"),
        "horizon_s":  horizon,
    }).sort_values("gain", ascending=False)
    all_importance[target] = imp

    # Store OOF predictions
    oof_predictions.loc[mask_val   & valid, f"pred_{target}"] = pred_va
    oof_predictions.loc[mask_test  & valid, f"pred_{target}"] = pred_te
    oof_predictions.loc[mask_val   & valid, f"actual_{target}"] = y_va
    oof_predictions.loc[mask_test  & valid, f"actual_{target}"] = y_te
    oof_predictions.loc[mask_val   & valid, "split"] = "val"
    oof_predictions.loc[mask_test  & valid, "split"] = "test"

# ---------------------------------------------------------------------------
# Step 6 — Daily IC breakdown (val + test)
# ---------------------------------------------------------------------------

log.info("=== Step 6: Daily IC breakdown ===")

daily_ic_rows = []
eval_mask = mask_val | mask_test
eval_df   = df[eval_mask].copy()

for horizon in HORIZONS:
    target = f"ret_{horizon}s"
    pred_col = f"pred_{target}"

    if pred_col not in oof_predictions.columns:
        continue

    eval_df2 = eval_df.merge(
        oof_predictions[["symbol", "ts_sec", pred_col]],
        on=["symbol", "ts_sec"], how="left"
    )

    for day, grp in eval_df2.groupby("date"):
        valid = grp[[target, pred_col]].dropna()
        if len(valid) < 50:
            continue
        ic, _ = spearmanr(valid[target], valid[pred_col])
        daily_ic_rows.append({
            "date":      str(day),
            "horizon_s": horizon,
            "ic":        round(float(ic), 4),
            "n":         len(valid),
            "split":     "val" if day in val_dates else "test",
        })

daily_ic = pd.DataFrame(daily_ic_rows)
if not daily_ic.empty:
    log.info("Daily IC summary:\n%s",
             daily_ic.groupby(["horizon_s", "split"])["ic"]
             .agg(["mean", "std", "min", "max"]).to_string())

# ---------------------------------------------------------------------------
# Step 7 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 7: Saving outputs ===")

# metrics.json
metrics_out = {
    "split_dates": {
        "train": [str(d) for d in train_dates],
        "val":   [str(d) for d in val_dates],
        "test":  [str(d) for d in test_dates],
    },
    "features": FEATURES,
    "lgb_params": LGB_PARAMS,
    "metrics": all_metrics,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
log.info("Saved metrics.json")

# feature_importance.csv
imp_df = pd.concat(all_importance.values(), ignore_index=True)
imp_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
log.info("Saved feature_importance.csv")

# oof_predictions.csv
oof_out = oof_predictions.dropna(subset=["split"])
oof_out.to_csv(OUT_DIR / "oof_predictions.csv", index=False)
log.info("Saved oof_predictions.csv  (%d rows)", len(oof_out))

# daily_ic.csv
if not daily_ic.empty:
    daily_ic.to_csv(OUT_DIR / "daily_ic.csv", index=False)
    log.info("Saved daily_ic.csv")

log.info("=== Done. Results in: %s ===", OUT_DIR)

# ---------------------------------------------------------------------------
# Step 8 — Print final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
for target, m in all_metrics.items():
    print(f"\n{target}:")
    for split in ["train", "val", "test"]:
        s = m[split]
        print(f"  {split:5s} | IC={s['ic']:+.4f}  RMSE={s['rmse']:.4f}  "
              f"Hit={s['hit_rate']:.4f}  n={s['n']:,}")

if not daily_ic.empty:
    print("\nDaily IC (val+test):")
    print(daily_ic.groupby(["horizon_s", "split"])["ic"]
          .agg(["mean", "std"]).round(4).to_string())
print("=" * 60)

01:10:17  INFO  === Step 1: Loading data and building features ===
01:10:18  INFO  Active contracts: ['26OCTFUT', '26NOVFUT']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

01:11:40  INFO  No files for USDINR26NOVFUT — skipping
01:11:40  INFO  Raw data: 369495 rows across symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT']
01:11:40  INFO  === Step 2: Building features ===
01:11:44  INFO  preprocess: 652039 rows after regularization
01:11:44  INFO  calculate_features: building CORE31 ...
01:12:01  INFO  build_core31_from_regularized: built 652039 rows, 31 CORE31 features
01:12:02  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
01:12:06  INFO  _build_book_geometry_features: built 652039 rows
01:12:06  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
01:12:07  INFO  _build_shock_recovery_features: bid_shocks=16201 ask_shocks=35165
01:12:07  INFO  _build_shock_recovery_features: built 652039 rows
01:12:07  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
01:12:09  INFO  _build_update_clustering_features: obs=369495 book_change=267711 bid=116198 ask=196885
01:12:10  INFO  _build_update_clusteri


FINAL RESULTS SUMMARY

ret_60s:
  train | IC=+0.3417  RMSE=0.7724  Hit=0.4702  n=356,901
  val   | IC=+0.2507  RMSE=0.9419  Hit=0.4821  n=127,868
  test  | IC=+0.2249  RMSE=0.7630  Hit=0.4591  n=90,107

ret_300s:
  train | IC=+0.3887  RMSE=1.6126  Hit=0.5924  n=353,462
  val   | IC=+0.1932  RMSE=2.0100  Hit=0.5606  n=126,668
  test  | IC=+0.1217  RMSE=1.6317  Hit=0.5311  n=89,305

Daily IC (val+test):
                   mean     std
horizon_s split                
60        test   0.2264  0.0615
          val    0.2512  0.0465
300       test   0.1270  0.0720
          val    0.1963  0.0607


In [12]:
import pandas as pd

imp = pd.read_csv(r"C:\projects\hedge_fund\results\lgbm_baseline\feature_importance.csv")
top15 = (
    imp[imp["horizon_s"] == 60]
    .sort_values("gain", ascending=False)
    .head(15)[["feature", "gain", "split"]]
    .reset_index(drop=True)
)
print(top15.to_string())

                             feature          gain  split
0                fragility_asymmetry  47687.575806     57
1                    log_rv_300s_new  30604.408936    109
2                impact_lambda_1800s  29093.073975    103
3                      clock_seconds  25702.573151     72
4             weighted_gap_asymmetry  24802.554047     65
5                    impact_r2_1800s  19023.271103     70
6             semivar_imbalance_900s  17954.804993     60
7                  ofi_activity_300s  17063.196045     57
8              sign_persistence_900s  16858.091881     52
9                     log_vr_30_300s  16400.695007     50
10               range_position_300s  14260.190086     44
11                     log_vr_5_300s  13087.385025     44
12  queue_replenishment_pressure_60s  12838.027031     53
13                 price_mom_60s_new  12741.140999     54
14                  ofi_activity_60s  10406.461945     36


In [13]:
"""
USDINR LightGBM Baseline — Return Forecasting
==============================================
Targets  : ret_60s, ret_300s (log-return in bps)
Features : BASE46 vs CHALLENGER50 comparison
Split    : Days 1-15 train | Days 16-20 val | Days 21-25 test
Outputs  : results/lgbm_baseline/metrics.json
           results/lgbm_baseline/feature_importance.csv
           results/lgbm_baseline/oof_predictions.csv
           results/lgbm_baseline/daily_ic.csv
           results/lgbm_baseline/comparison.csv
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.contracts import active_contracts
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import BASE46, CHALLENGER50, CHALLENGERS4

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR = pathlib.Path(r"C:\projects\hedge_fund\results\lgbm_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SETS = {
    "base46":       BASE46,
    "challenger50": CHALLENGER50,
}
HORIZONS   = [60, 300]
TRAIN_DAYS = 15
VAL_DAYS   = 5

LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     100,
    "n_estimators":          1000,
    "early_stopping_rounds": 50,
}

# ---------------------------------------------------------------------------
# Step 1 — Load data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df["symbol"] = sym
            all_frames.append(df)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded — check S3 paths and contracts")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build all 50 features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d columns", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Forward return targets
# ---------------------------------------------------------------------------

log.info("=== Step 3: Building targets ===")

df = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
for h in HORIZONS:
    df[f"ret_{h}s"] = (g["log_mid"].shift(-h) - df["log_mid"]) * 1e4
    log.info("Target ret_%ds: %.1f%% non-null", h, df[f"ret_{h}s"].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 4 — Train/val/test split
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting ===")

all_dates   = sorted(df["date"].unique())
n_days      = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]

log.info("Train: %d days  Val: %d days  Test: %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 5 — Train and evaluate both feature sets
# ---------------------------------------------------------------------------

log.info("=== Step 5: Training BASE46 vs CHALLENGER50 ===")

all_metrics    = {}
all_importance = {}
comparison_rows = []

oof = df[["symbol", "ts_sec", "date"]].copy()
oof["split"] = np.where(mask_val, "val", np.where(mask_test, "test", "train"))

def compute_metrics(y_true, y_pred, label):
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    ic, _ = spearmanr(y_true, y_pred)
    hit   = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    log.info("%s | IC=%+.4f  RMSE=%.4f  Hit=%.4f  n=%d",
             label, ic, rmse, hit, len(y_true))
    return {"ic": ic, "rmse": rmse, "hit_rate": hit, "n": len(y_true)}


for feat_name, features in FEATURE_SETS.items():
    log.info("--- Feature set: %s (%d features) ---", feat_name, len(features))

    for horizon in HORIZONS:
        target  = f"ret_{horizon}s"
        run_key = f"{feat_name}/{target}"

        valid = df[features + [target]].notna().all(axis=1)

        tr = df[mask_train & valid]
        va = df[mask_val   & valid]
        te = df[mask_test  & valid]

        log.info("%s | train=%d val=%d test=%d", run_key, len(tr), len(va), len(te))

        X_tr, y_tr = tr[features].values, tr[target].values
        X_va, y_va = va[features].values, va[target].values
        X_te, y_te = te[features].values, te[target].values

        ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
        ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

        params = {k: v for k, v in LGB_PARAMS.items()
                  if k not in ("n_estimators", "early_stopping_rounds")}

        model = lgb.train(
            params,
            ds_tr,
            num_boost_round = LGB_PARAMS["n_estimators"],
            valid_sets      = [ds_va],
            callbacks       = [
                lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
                lgb.log_evaluation(200),
            ],
        )

        pred_tr = model.predict(X_tr)
        pred_va = model.predict(X_va)
        pred_te = model.predict(X_te)

        metrics = {
            "train": compute_metrics(y_tr, pred_tr, f"{run_key}/train"),
            "val":   compute_metrics(y_va, pred_va, f"{run_key}/val"),
            "test":  compute_metrics(y_te, pred_te, f"{run_key}/test"),
            "best_iteration": model.best_iteration,
        }
        all_metrics[run_key] = metrics

        # Feature importance
        imp = pd.DataFrame({
            "feature":     features,
            "gain":        model.feature_importance("gain"),
            "split_count": model.feature_importance("split"),
            "feat_set":    feat_name,
            "horizon_s":   horizon,
            "is_challenger": [f in CHALLENGERS4 for f in features],
        }).sort_values("gain", ascending=False)
        all_importance[run_key] = imp

        # OOF predictions
        pred_col  = f"pred_{feat_name}_{target}"
        actual_col = f"actual_{target}"
        oof.loc[va.index, pred_col]  = pred_va
        oof.loc[te.index, pred_col]  = pred_te
        oof.loc[va.index, actual_col] = y_va
        oof.loc[te.index, actual_col] = y_te

        # Comparison row
        comparison_rows.append({
            "feat_set":   feat_name,
            "n_features": len(features),
            "horizon_s":  horizon,
            "train_ic":   round(metrics["train"]["ic"], 4),
            "val_ic":     round(metrics["val"]["ic"],   4),
            "test_ic":    round(metrics["test"]["ic"],  4),
            "test_rmse":  round(metrics["test"]["rmse"], 4),
            "test_hit":   round(metrics["test"]["hit_rate"], 4),
            "best_iter":  model.best_iteration,
        })

# ---------------------------------------------------------------------------
# Step 6 — Challenger feature contribution
# ---------------------------------------------------------------------------

log.info("=== Step 6: Challenger feature contribution ===")

for horizon in HORIZONS:
    base_key  = f"base46/ret_{horizon}s"
    chal_key  = f"challenger50/ret_{horizon}s"
    base_ic   = all_metrics[base_key]["test"]["ic"]
    chal_ic   = all_metrics[chal_key]["test"]["ic"]
    delta     = chal_ic - base_ic
    pct       = delta / abs(base_ic) * 100
    log.info("ret_%ds | BASE46 IC=%.4f  CHALLENGER50 IC=%.4f  delta=%+.4f (%+.1f%%)",
             horizon, base_ic, chal_ic, delta, pct)

    # Rank of each challenger feature in the challenger50 model
    imp50 = all_importance[chal_key].reset_index(drop=True)
    imp50["rank"] = imp50.index + 1
    chal_imp = imp50[imp50["is_challenger"]][["feature", "gain", "rank"]]
    log.info("Challenger feature ranks (ret_%ds):\n%s", horizon, chal_imp.to_string(index=False))

# ---------------------------------------------------------------------------
# Step 7 — Daily IC breakdown
# ---------------------------------------------------------------------------

log.info("=== Step 7: Daily IC breakdown ===")

daily_ic_rows = []
eval_mask = mask_val | mask_test

for feat_name, features in FEATURE_SETS.items():
    for horizon in HORIZONS:
        target   = f"ret_{horizon}s"
        pred_col = f"pred_{feat_name}_{target}"
        act_col  = f"actual_{target}"

        if pred_col not in oof.columns:
            continue

        sub = oof[eval_mask & oof[pred_col].notna() & oof[act_col].notna()].copy()

        for day, grp in sub.groupby("date"):
            if len(grp) < 50:
                continue
            ic, _ = spearmanr(grp[act_col], grp[pred_col])
            daily_ic_rows.append({
                "date":      str(day),
                "feat_set":  feat_name,
                "horizon_s": horizon,
                "ic":        round(float(ic), 4),
                "n":         len(grp),
                "split":     "val" if day in val_dates else "test",
            })

daily_ic = pd.DataFrame(daily_ic_rows)

# ---------------------------------------------------------------------------
# Step 8 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 8: Saving outputs ===")

# metrics.json
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({
        "split_dates": {
            "train": [str(d) for d in train_dates],
            "val":   [str(d) for d in val_dates],
            "test":  [str(d) for d in test_dates],
        },
        "lgb_params": LGB_PARAMS,
        "metrics":    all_metrics,
    }, f, indent=2)
log.info("Saved metrics.json")

# feature_importance.csv
imp_df = pd.concat(all_importance.values(), ignore_index=True)
imp_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
log.info("Saved feature_importance.csv")

# oof_predictions.csv
oof_out = oof[eval_mask].copy()
oof_out.to_csv(OUT_DIR / "oof_predictions.csv", index=False)
log.info("Saved oof_predictions.csv  (%d rows)", len(oof_out))

# daily_ic.csv
if not daily_ic.empty:
    daily_ic.to_csv(OUT_DIR / "daily_ic.csv", index=False)
    log.info("Saved daily_ic.csv")

# comparison.csv — key output
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
log.info("Saved comparison.csv")

# ---------------------------------------------------------------------------
# Step 9 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("BASE46  vs  CHALLENGER50  —  COMPARISON SUMMARY")
print("=" * 70)
print(comparison.to_string(index=False))

print("\n--- Challenger IC uplift ---")
for horizon in HORIZONS:
    b = comparison[(comparison["feat_set"] == "base46")      & (comparison["horizon_s"] == horizon)]["test_ic"].values[0]
    c = comparison[(comparison["feat_set"] == "challenger50") & (comparison["horizon_s"] == horizon)]["test_ic"].values[0]
    print(f"  ret_{horizon}s:  BASE46={b:+.4f}  CHALLENGER50={c:+.4f}  delta={c-b:+.4f}")

if not daily_ic.empty:
    print("\n--- Daily IC (test days only) ---")
    print(
        daily_ic[daily_ic["split"] == "test"]
        .groupby(["feat_set", "horizon_s"])["ic"]
        .agg(["mean", "std", "min"])
        .round(4)
        .to_string()
    )
print("=" * 70)

23:49:46  INFO  === Step 1: Loading data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (4 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

23:51:18  INFO  Raw: 392595 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
23:51:18  INFO  === Step 2: Building features ===
23:51:23  INFO  preprocess: 709627 rows after regularization
23:51:23  INFO  calculate_features: building CORE31 ...
23:51:54  INFO  build_core31_from_regularized: built 709627 rows, 31 CORE31 features
23:51:56  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
23:52:04  INFO  _build_book_geometry_features: built 709627 rows
23:52:04  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
23:52:06  INFO  _build_shock_recovery_features: bid_shocks=17240 ask_shocks=36675
23:52:07  INFO  _build_shock_recovery_features: built 709627 rows
23:52:07  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
23:52:09  INFO  _build_update_clustering_features: obs=392595 book_change=279430 bid=122343 ask=204364
23:52:11  INFO  _build_update_clustering_features: built 709627 rows
23:52:11  INFO  


BASE46  vs  CHALLENGER50  —  COMPARISON SUMMARY
    feat_set  n_features  horizon_s  train_ic  val_ic  test_ic  test_rmse  test_hit  best_iter
      base46          46         60    0.3417  0.2507   0.2211     0.8058    0.4540         46
      base46          46        300    0.3887  0.1932   0.1089     1.7338    0.5218         24
challenger50          50         60    0.3603  0.2476   0.2322     0.8494    0.4718         39
challenger50          50        300    0.3852  0.1897   0.1054     1.8065    0.5230         15

--- Challenger IC uplift ---
  ret_60s:  BASE46=+0.2211  CHALLENGER50=+0.2322  delta=+0.0111
  ret_300s:  BASE46=+0.1089  CHALLENGER50=+0.1054  delta=-0.0035

--- Daily IC (test days only) ---
                          mean     std     min
feat_set     horizon_s                        
base46       60         0.2253  0.0496  0.1414
             300        0.1196  0.0591  0.0220
challenger50 60         0.2315  0.0637  0.1486
             300        0.1316  0.0498  0.0805


In [14]:
import pandas as pd

imp = pd.read_csv(r"C:\projects\hedge_fund\results\lgbm_baseline\feature_importance.csv")

challengers = ["recovery_pressure", "ask_shock_age", 
               "side_update_imbalance_60", "migration_activity_asym_60"]

# Rank within challenger50 model, 60s horizon
imp60 = (
    imp[(imp["feat_set"] == "challenger50") & (imp["horizon_s"] == 60)]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
imp60["rank"] = imp60.index + 1

print("=== Challenger features rank (60s, out of 50) ===")
print(imp60[imp60["feature"].isin(challengers)][["rank", "feature", "gain"]].to_string(index=False))

print("\n=== Top 15 features overall (challenger50, 60s) ===")
print(imp60.head(15)[["rank", "feature", "gain"]].to_string(index=False))

=== Challenger features rank (60s, out of 50) ===
 rank                    feature        gain
   14 migration_activity_asym_60 8455.507057
   34          recovery_pressure 2188.201988
   36   side_update_imbalance_60 2131.235008
   44              ask_shock_age  477.005005

=== Top 15 features overall (challenger50, 60s) ===
 rank                          feature         gain
    1              fragility_asymmetry 46415.743179
    2              impact_lambda_1800s 26203.009972
    3                  log_rv_300s_new 22781.806007
    4                    clock_seconds 21297.891930
    5            sign_persistence_900s 16406.730026
    6                  impact_r2_1800s 16067.728043
    7           weighted_gap_asymmetry 15752.697006
    8                   log_vr_30_300s 13116.731964
    9                price_mom_60s_new 12303.076996
   10           semivar_imbalance_900s 11204.032097
   11 queue_replenishment_pressure_60s 10918.634048
   12                ofi_activity_300s 10770.443